In [5]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path

LAST_NAME = 'Magalong'
STUDENT_ID = 'TUPM-26-1134'

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

print('Seed:', SEED_DIGIT)
print('ID Sum:', ID_SUM)
print('Name Length:', NAME_LENGTH)

Seed: 4
ID Sum: 17
Name Length: 8


In [6]:
# [CELL 2] CREATE ACTIVITY WORKSPACE
WORK_DIR = Path('.')
DATA_DIR = WORK_DIR / 'data'
OUTPUT_DIR = WORK_DIR / 'output'
BACKUP_DIR = WORK_DIR / 'backup'

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print('Student:', LAST_NAME)
print('Student ID:', STUDENT_ID)
print('Workspace:', WORK_DIR.resolve())
print('Data folder exists:', DATA_DIR.exists())
print('Output folder exists:', OUTPUT_DIR.exists())
print('Backup folder exists:', BACKUP_DIR.exists())

Student: Magalong
Student ID: TUPM-26-1134
Workspace: C:\Users\Paul\Documents\TUPM-BSECE\1C\AECEL1\Lab\Lab6_File_Handling_Magalong\Lab6_Additional_Exercises
Data folder exists: True
Output folder exists: True
Backup folder exists: True


In [7]:
# [CELL 3] EXERCISE 2 PARAMETERS
import csv
import json
import shutil

dataset_path = DATA_DIR / 'user_behavior_dataset.csv'

target_os = 'Android' if SEED_DIGIT % 2 == 0 else 'iOS'
target_class = (SEED_DIGIT % 5) + 1

print('Dataset:', dataset_path)
print('Dataset exists:', dataset_path.exists())
print('Target operating system:', target_os)
print('Target user behavior class:', target_class)

Dataset: data\user_behavior_dataset.csv
Dataset exists: True
Target operating system: Android
Target user behavior class: 5


In [8]:
# [CELL 4] READ DATASET AND FILTER RECORDS
export_fields = [
    'User ID',
    'Device Model',
    'Operating System',
    'Battery Drain (mAh/day)',
    'Data Usage (MB/day)'
]

total_records = 0
os_records = 0
invalid_records = 0
selected_users = []

with open(dataset_path, 'r', encoding='utf-8-sig', newline='') as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_records += 1

        if row['Operating System'].strip().lower() != target_os.lower():
            continue

        os_records += 1

        try:
            behavior_class = int(row['User Behavior Class'])
            battery = float(row['Battery Drain (mAh/day)'])
            data_usage = float(row['Data Usage (MB/day)'])
        except (KeyError, ValueError):
            invalid_records += 1
            continue

        if behavior_class == target_class:
            selected_users.append({
                'User ID': row['User ID'].strip(),
                'Device Model': row['Device Model'].strip(),
                'Operating System': row['Operating System'].strip(),
                'Battery Drain (mAh/day)': battery,
                'Data Usage (MB/day)': data_usage
            })

print('Total records in dataset:', total_records)
print('Records matching operating system:', os_records)
print('Records in target behavior class:', len(selected_users))
print('Invalid records skipped:', invalid_records)

Total records in dataset: 700
Records matching operating system: 554
Records in target behavior class: 105
Invalid records skipped: 0


In [9]:
# [CELL 5] CALCULATE AVERAGES
if selected_users:
    average_battery = sum(u['Battery Drain (mAh/day)'] for u in selected_users) / len(selected_users)
    average_data = sum(u['Data Usage (MB/day)'] for u in selected_users) / len(selected_users)
    print('Average battery drain (mAh/day):', round(average_battery, 2))
    print('Average data usage (MB/day):', round(average_data, 2))
else:
    average_battery = None
    average_data = None
    print('No matching records found. Averages are not available.')

Average battery drain (mAh/day): 2693.06
Average data usage (MB/day): 1984.77


In [10]:
# [CELL 6] EXPORT SELECTED USERS TO CSV
exported_csv_path = OUTPUT_DIR / f'{LAST_NAME}_{STUDENT_ID}_mobile_users.csv'

with open(exported_csv_path, 'w', encoding='utf-8', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=export_fields)
    writer.writeheader()
    writer.writerows(selected_users)

print('Created:', exported_csv_path)
print('File size:', exported_csv_path.stat().st_size, 'bytes')

Created: output\Magalong_TUPM-26-1134_mobile_users.csv
File size: 4379 bytes


In [11]:
# [CELL 7] VERIFY ROW COUNT AND WRITE JSON SUMMARY
with open(exported_csv_path, 'r', encoding='utf-8', newline='') as file:
    exported_rows = sum(1 for _ in csv.reader(file)) - 1

count_verified = exported_rows == len(selected_users)

summary_path = OUTPUT_DIR / f'{LAST_NAME}_{STUDENT_ID}_mobile_summary.json'

summary = {
    'last_name': LAST_NAME,
    'student_id': STUDENT_ID,
    'seed_digit': SEED_DIGIT,
    'operating_system': target_os,
    'user_behavior_class': target_class,
    'total_records': total_records,
    'records_matching_os': os_records,
    'records_in_target_class': len(selected_users),
    'average_battery_drain_mah_per_day': round(average_battery, 2) if average_battery is not None else None,
    'average_data_usage_mb_per_day': round(average_data, 2) if average_data is not None else None,
    'exported_row_count': exported_rows,
    'count_verified': count_verified
}

with open(summary_path, 'w', encoding='utf-8') as file:
    json.dump(summary, file, indent=4)

print('Created:', summary_path)
print('Exported rows:', exported_rows)
print('Selected records:', len(selected_users))
print('Count verification:', 'VERIFIED' if count_verified else 'CHECK')

Created: output\Magalong_TUPM-26-1134_mobile_summary.json
Exported rows: 105
Selected records: 105
Count verification: VERIFIED


In [12]:
# [CELL 8] BACKUP EXPORTED CSV AND REPORT SIZE
backup_path = BACKUP_DIR / exported_csv_path.name

shutil.copy2(exported_csv_path, backup_path)

backup_size = backup_path.stat().st_size

print('Original:', exported_csv_path)
print('Backup:', backup_path)
print('Backup exists:', backup_path.exists())
print('Backup size:', backup_size, 'bytes')

Original: output\Magalong_TUPM-26-1134_mobile_users.csv
Backup: backup\Magalong_TUPM-26-1134_mobile_users.csv
Backup exists: True
Backup size: 4379 bytes


In [13]:
# [CELL 9] FINAL OUTPUT
print('ASSESSMENT DATA - MOBILE DEVICE USAGE PROFILING')
print('-----------------------------------------------')
print('Operating system and behavior class:', target_os, '/ Class', target_class)
print('Total records in dataset:', total_records)
print('Records matching operating system:', os_records)
print('Records in target behavior class:', len(selected_users))
print('Average battery drain (mAh/day):', round(average_battery, 2) if average_battery is not None else 'N/A')
print('Average data usage (MB/day):', round(average_data, 2) if average_data is not None else 'N/A')
print('Count verification:', 'VERIFIED' if count_verified else 'CHECK')
print('Exported CSV file:', exported_csv_path.name)
print('JSON summary file:', summary_path.name)
print('Backup file and size:', backup_path.name, f'({backup_size} bytes)')

ASSESSMENT DATA - MOBILE DEVICE USAGE PROFILING
-----------------------------------------------
Operating system and behavior class: Android / Class 5
Total records in dataset: 700
Records matching operating system: 554
Records in target behavior class: 105
Average battery drain (mAh/day): 2693.06
Average data usage (MB/day): 1984.77
Count verification: VERIFIED
Exported CSV file: Magalong_TUPM-26-1134_mobile_users.csv
JSON summary file: Magalong_TUPM-26-1134_mobile_summary.json
Backup file and size: Magalong_TUPM-26-1134_mobile_users.csv (4379 bytes)
